In [ ]:

# ==========================================
# GOOGLE COLAB GPU DIAGNOSTIC
# ==========================================

import os
import subprocess
import torch
import time

print("=" * 55)
print("          COLAB GPU DIAGNOSTIC REPORT")
print("=" * 55)

# 1. GPU availability
print("\n[1] GPU Availability")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    # 2. GPU information
    print("\n[2] GPU Information")
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())
    print("CUDA Version:", torch.version.cuda)

    props = torch.cuda.get_device_properties(0)
    print("Total GPU Memory:",
          round(props.total_memory / 1024**3, 2), "GB")

    # 3. GPU memory status
    print("\n[3] GPU Memory")
    print("Allocated:",
          round(torch.cuda.memory_allocated(0) / 1024**3, 3), "GB")
    print("Reserved:",
          round(torch.cuda.memory_reserved(0) / 1024**3, 3), "GB")

    # 4. Actual CUDA computation test
    print("\n[4] CUDA Computation Test")

    try:
        a = torch.randn(3000, 3000, device="cuda")
        b = torch.randn(3000, 3000, device="cuda")

        start = time.time()
        c = torch.matmul(a, b)
        torch.cuda.synchronize()
        elapsed = time.time() - start

        print("Matrix multiplication: SUCCESS")
        print("Output device:", c.device)
        print("Execution time:", round(elapsed, 4), "seconds")

    except Exception as e:
        print("CUDA computation FAILED:", str(e))

    # 5. NVIDIA status
    print("\n[5] NVIDIA GPU Status")
    subprocess.run(["nvidia-smi"])

    print("\nGPU DIAGNOSTIC COMPLETED")

else:
    print("\n[ERROR] CUDA GPU NOT AVAILABLE")
    print("Go to Runtime > Change runtime type")
    print("Select T4 GPU (or another available GPU)")
    print("Save and reconnect.")

print("\n" + "=" * 55)


          COLAB GPU DIAGNOSTIC REPORT

[1] GPU Availability
PyTorch version: 2.14.1+cu130
CUDA available: True

[2] GPU Information
GPU Name: Tesla T4
GPU Count: 1
CUDA Version: 13.0
Total GPU Memory: 14.56 GB

[3] GPU Memory
Allocated: 0.0 GB
Reserved: 0.0 GB

[4] CUDA Computation Test
Matrix multiplication: SUCCESS
Output device: cuda:0
Execution time: 0.0854 seconds

[5] NVIDIA GPU Status

GPU DIAGNOSTIC COMPLETED



In [ ]:
import torch, os

print("GPU:", torch.cuda.get_device_name(0))
print("GPU Memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")
print("CUDA:", torch.version.cuda)

!nvidia-smi

GPU: Tesla T4
GPU Memory: 14.56 GB
CUDA: 13.0
Wed Nov 25 13:09:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             26W /   70W |     245MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-

In [ ]:

import os
import subprocess
from pathlib import Path

REPO = Path("/content/cartoonization-robustness-study")
BRANCH = "research/cafr-net-phase-iv"

if not REPO.exists():
    subprocess.run([
        "git", "clone", "-b", BRANCH,
        "https://github.com/Karthik6-cpu/cartoonization-robustness-study.git",
        str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "fetch", "origin"], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=REPO, check=True)

SOURCE = REPO / "research/CAFRNet_v2/source"

print("Repository ready:", REPO)
print("Experiment launcher:", (SOURCE / "run_reviewer_matrix.sh").exists())

print("\nAvailable experiment scripts:")
for f in SOURCE.glob("*"):
    print(f.name)

print("\nGPU:", subprocess.getoutput("nvidia-smi -L"))


Repository ready: /content/cartoonization-robustness-study
Experiment launcher: True

Available experiment scripts:
train_cafr_net.py
train_architecture.py
legacy
evaluate_cafr_net.py
cafr_net.py
evaluate_architecture.py
run_reviewer_matrix.sh
requirements.txt
analyze_architecture_comparison.py

GPU: GPU 0: Tesla T4 (UUID: GPU-3229ee6c-8c75-6207-6c7b-2b20dd471b28)


In [ ]:

import subprocess, sys, torch
from pathlib import Path

REPO = Path("/content/cartoonization-robustness-study")
SOURCE = REPO / "research/CAFRNet_v2/source"

print("Installing research dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "opencv-python-headless", "pandas", "scikit-image",
    "matplotlib", "pillow"
], check=True)

print("\nGPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())
print("PyTorch:", torch.__version__)

print("\nChecking datasets:")
for p in [
    Path("/content/cafr_data"),
    Path("/content/oxford_pet_data")
]:
    print(p, "READY" if (p/"train").exists()
          and (p/"val").exists() else "NOT PREPARED")

print("\nExisting experiment checkpoints:")
for root in [
    Path("/content/cafr_runs/imagenette"),
    Path("/content/oxford_runs/oxford")
]:
    if root.exists():
        for f in root.rglob("latest_model.pt"):
            print(f)

print("\nExperiment launcher:", (SOURCE/"run_reviewer_matrix.sh").exists())
print("\nENVIRONMENT PREPARATION COMPLETE")


Installing research dependencies...

GPU: Tesla T4
CUDA: True
PyTorch: 2.14.1+cu130

Checking datasets:
/content/cafr_data NOT PREPARED
/content/oxford_pet_data NOT PREPARED

Existing experiment checkpoints:

Experiment launcher: True

ENVIRONMENT PREPARATION COMPLETE


In [ ]:

import os, tarfile, urllib.request, random, shutil
from pathlib import Path

from sklearn.model_selection import train_test_split

ROOT = Path("/content/cafr_data")
ARCHIVE = Path("/content/imagenette2.tgz")
URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz"

print("1. Downloading Imagenette2...")
if not ARCHIVE.exists():
    urllib.request.urlretrieve(URL, ARCHIVE)

print("2. Extracting dataset...")
EXTRACT = Path("/content/imagenette2")
if not EXTRACT.exists():
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall("/content")

SRC = EXTRACT
if not (SRC/"train").exists():
    raise FileNotFoundError("Imagenette train folder not found.")

print("3. Creating fixed train/validation split...")
if ROOT.exists():
    shutil.rmtree(ROOT)

for split in ["train", "val", "test_original"]:
    (ROOT/split).mkdir(parents=True, exist_ok=True)

random.seed(42)

for class_dir in sorted((SRC/"train").iterdir()):
    if not class_dir.is_dir():
        continue

    images = sorted([
        p for p in class_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

    train_files, val_files = train_test_split(
        images, test_size=0.10, random_state=42
    )

    for split, files in [("train", train_files), ("val", val_files)]:
        target = ROOT/split/class_dir.name
        target.mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.copy2(f, target/f.name)

    official_test = SRC/"val"/class_dir.name
    target = ROOT/"test_original"/class_dir.name
    target.mkdir(parents=True, exist_ok=True)

    for f in official_test.iterdir():
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            shutil.copy2(f, target/f.name)

print("\nDATASET PREPARATION COMPLETE")
for split in ["train", "val", "test_original"]:
    count = sum(1 for p in (ROOT/split).rglob("*") if p.is_file())
    print(split, ":", count, "images")


1. Downloading Imagenette2...
2. Extracting dataset...


/tmp/ipykernel_485/4009793122.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/content")


3. Creating fixed train/validation split...

DATASET PREPARATION COMPLETE
train : 8516 images
val : 953 images
test_original : 3925 images


In [ ]:

import urllib.request, torch, hashlib, csv
from pathlib import Path

BASE = "https://github.com/Karthik6-cpu/cartoonization-robustness-study/releases/download/cafrnet-v2-checkpoints-2026-11-25"

ROOT = Path("/content/cafr_runs/imagenette")
SEED42 = ROOT / "cafrnet_seed42"
SEED123 = ROOT / "cafrnet_seed123"

SEED42.mkdir(parents=True, exist_ok=True)
SEED123.mkdir(parents=True, exist_ok=True)

downloads = [
    (f"{BASE}/best_model.pt", SEED42/"best_model.pt"),
    (f"{BASE}/cafrnet_v2_seed123_latest_model.pt",
     SEED123/"latest_model.pt"),
    (f"{BASE}/cafrnet_v2_seed123_best_model.pt",
     SEED123/"best_model.pt")
]

for url, path in downloads:
    if not path.exists():
        print("Downloading:", path.name)
        urllib.request.urlretrieve(url, path)
    else:
        print("Already available:", path.name)

def manifest_hash(folder):
    folder = Path(folder)
    rows = []
    for class_dir in sorted(folder.iterdir()):
        if class_dir.is_dir():
            for f in class_dir.iterdir():
                if f.is_file():
                    rows.append(
                        (f"{class_dir.name}/{f.name}", class_dir.name)
                    )
    rows.sort()
    raw = "\n".join(f"{p}\t{y}" for p,y in rows).encode()
    return hashlib.sha256(raw).hexdigest()

train_hash = manifest_hash("/content/cafr_data/train")
val_hash = manifest_hash("/content/cafr_data/val")

print("\nCURRENT DATASET HASHES")
print("Train:", train_hash)
print("Validation:", val_hash)

for label, path in [
    ("Seed 42", SEED42/"best_model.pt"),
    ("Seed 123", SEED123/"latest_model.pt")
]:
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = ck.get("metadata", {})
    print("\n", label)
    print("Architecture:", ck.get("architecture"))
    print("Seed:", ck.get("seed"))
    print("Checkpoint epoch:", ck.get("epoch"))
    print("Saved train hash:", meta.get("train_manifest_sha256"))
    print("Saved validation hash:", meta.get("validation_manifest_sha256"))
    print("Train split matches:",
          meta.get("train_manifest_sha256") == train_hash)
    print("Validation split matches:",
          meta.get("validation_manifest_sha256") == val_hash)

print("\nRESTORE AND COMPATIBILITY CHECK COMPLETE")


Downloading: best_model.pt
Downloading: latest_model.pt
Downloading: best_model.pt

CURRENT DATASET HASHES
Train: 90445c7b9af9849eb6788fb90cbb78a4a788b49225d9db32a9cf1be583f3e715
Validation: e722efd4e4cbc6ec89415f40972d161420b5bedc9140e3fdf9654b65724f9bbf

 Seed 42
Architecture: cafrnet_v2
Seed: 42
Checkpoint epoch: 3
Saved train hash: 39438146a850a0021c69e7d8fe215f20a6c883185f42e036387ff9627ca00d12
Saved validation hash: fd16a292e5810e9455d786f397ee69a2a8b514c7c29a2f51400673cbc3a36a6f
Train split matches: False
Validation split matches: False

 Seed 123
Architecture: cafrnet_v2
Seed: 123
Checkpoint epoch: 1
Saved train hash: 39438146a850a0021c69e7d8fe215f20a6c883185f42e036387ff9627ca00d12
Saved validation hash: fd16a292e5810e9455d786f397ee69a2a8b514c7c29a2f51400673cbc3a36a6f
Train split matches: False
Validation split matches: False

RESTORE AND COMPATIBILITY CHECK COMPLETE


In [ ]:

import pandas as pd
import shutil, hashlib
from pathlib import Path
import torch

REPO = Path("/content/cartoonization-robustness-study")
SRC = Path("/content/imagenette2")
BACKUP = REPO / "research/CAFRNet_v2/drive_backup_2026-11-25"
MANIFEST = BACKUP / "imagenette_controlled_split.csv"
ROOT = Path("/content/cafr_data_matched")

assert MANIFEST.exists(), "Original split manifest missing"
assert SRC.exists(), "Extracted Imagenette dataset missing"

df = pd.read_csv(MANIFEST)
print("Original split counts:")
print(df["split"].value_counts())

if ROOT.exists():
    shutil.rmtree(ROOT)

for split in ["train", "val"]:
    (ROOT / split).mkdir(parents=True, exist_ok=True)

missing = []

for row in df.itertuples(index=False):
    split = row.split
    cls = row.class_name
    rel = Path(row.relative_path)

    candidates = [
        SRC / "train" / rel,
        SRC / "val" / rel
    ]
    source = next((p for p in candidates if p.is_file()), None)

    if source is None:
        missing.append(str(rel))
        continue

    dest = ROOT / split / cls / source.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)

if missing:
    print("Missing files:", len(missing))
    print(missing[:10])
    raise RuntimeError("Cannot reconstruct exact original split.")

def folder_hash(folder):
    rows = []
    for cls in sorted(Path(folder).iterdir()):
        if cls.is_dir():
            for f in cls.iterdir():
                if f.is_file():
                    rows.append((f"{cls.name}/{f.name}", cls.name))
    rows.sort()
    raw = "\n".join(f"{p}\t{y}" for p,y in rows).encode()
    return hashlib.sha256(raw).hexdigest()

train_hash = folder_hash(ROOT / "train")
val_hash = folder_hash(ROOT / "val")

ck42 = torch.load(
    "/content/cafr_runs/imagenette/cafrnet_seed42/best_model.pt",
    map_location="cpu", weights_only=False
)
ck123 = torch.load(
    "/content/cafr_runs/imagenette/cafrnet_seed123/latest_model.pt",
    map_location="cpu", weights_only=False
)

expected_train = ck42["metadata"]["train_manifest_sha256"]
expected_val = ck42["metadata"]["validation_manifest_sha256"]

print("\nRestored train hash:", train_hash)
print("Restored validation hash:", val_hash)
print("Expected train hash:", expected_train)
print("Expected validation hash:", expected_val)

MATCH = train_hash == expected_train and val_hash == expected_val

print("\nEXACT ORIGINAL SPLIT MATCH:", MATCH)

if not MATCH:
    raise RuntimeError(
        "Manifest files do not reproduce checkpoint split. "
        "Do not resume training with these checkpoints."
    )

print("\nSUCCESS: Original checkpoint split restored.")
print("Seed 42 checkpoint epoch:", ck42["epoch"])
print("Seed 123 checkpoint epoch:", ck123["epoch"])
print("Seed 123 can now be resumed on the matching split.")


Original split counts:
split
train         8048
test          3925
validation    1421
Name: count, dtype: int64

Restored train hash: 39438146a850a0021c69e7d8fe215f20a6c883185f42e036387ff9627ca00d12
Restored validation hash: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
Expected train hash: 39438146a850a0021c69e7d8fe215f20a6c883185f42e036387ff9627ca00d12
Expected validation hash: fd16a292e5810e9455d786f397ee69a2a8b514c7c29a2f51400673cbc3a36a6f

EXACT ORIGINAL SPLIT MATCH: False


RuntimeError: Manifest files do not reproduce checkpoint split. Do not resume training with these checkpoints.

In [ ]:

import pandas as pd
import shutil, hashlib
from pathlib import Path

SRC = Path("/content/imagenette2")
BACKUP = Path("/content/cartoonization-robustness-study/research/CAFRNet_v2/drive_backup_2026-11-25")
ROOT = Path("/content/cafr_data_matched")

df = pd.read_csv(BACKUP / "imagenette_controlled_split.csv")

# Preserve the already-correct training split.
val_dir = ROOT / "val"
if val_dir.exists():
    shutil.rmtree(val_dir)
val_dir.mkdir(parents=True)

# Restore manifest's "validation" rows into the trainer's "val" folder.
val_rows = df[df["split"] == "validation"]

for row in val_rows.itertuples(index=False):
    rel = Path(row.relative_path)
    candidates = [SRC / "train" / rel, SRC / "val" / rel]
    source = next((p for p in candidates if p.is_file()), None)

    if source is None:
        raise FileNotFoundError(str(rel))

    dest = val_dir / row.class_name / source.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)

def folder_hash(folder):
    rows = []
    for cls in sorted(Path(folder).iterdir()):
        if cls.is_dir():
            for f in cls.iterdir():
                if f.is_file():
                    rows.append((f"{cls.name}/{f.name}", cls.name))
    rows.sort()
    raw = "\n".join(f"{p}\t{y}" for p,y in rows).encode()
    return hashlib.sha256(raw).hexdigest()

train_hash = folder_hash(ROOT / "train")
val_hash = folder_hash(ROOT / "val")

import torch
ck = torch.load(
    "/content/cafr_runs/imagenette/cafrnet_seed42/best_model.pt",
    map_location="cpu", weights_only=False
)

print("Training images:", sum(1 for p in (ROOT/"train").rglob("*") if p.is_file()))
print("Validation images:", sum(1 for p in (ROOT/"val").rglob("*") if p.is_file()))

print("Train match:",
      train_hash == ck["metadata"]["train_manifest_sha256"])
print("Validation match:",
      val_hash == ck["metadata"]["validation_manifest_sha256"])

MATCH = (
    train_hash == ck["metadata"]["train_manifest_sha256"]
    and val_hash == ck["metadata"]["validation_manifest_sha256"]
)

print("\nEXACT ORIGINAL SPLIT MATCH:", MATCH)


Training images: 8048
Validation images: 1421
Train match: True
Validation match: True

EXACT ORIGINAL SPLIT MATCH: True


In [ ]:

from pathlib import Path
import subprocess

BASE = Path("/content/cartoonization-robustness-study/research/CAFRNet_v2")

print("=== REVIEWER MATRIX LAUNCHER ===")
launcher = BASE / "run_reviewer_matrix.sh"
if launcher.exists():
    print(launcher.read_text())
else:
    print("Launcher not found at:", launcher)

for name in ["train_cafr_net.py", "train_architecture.py"]:
    script = BASE / name
    print(f"\n=== {name} ARGUMENTS ===")
    if script.exists():
        result = subprocess.run(
            ["python", str(script), "--help"],
            cwd=str(BASE), capture_output=True, text=True
        )
        print(result.stdout[:12000])
        if result.stderr:
            print("STDERR:", result.stderr[:2000])
    else:
        print("Not found:", script)

print("\n=== AVAILABLE CHECKPOINTS ===")
for p in [
    Path("/content/cafr_runs/imagenette/cafrnet_seed42/best_model.pt"),
    Path("/content/cafr_runs/imagenette/cafrnet_seed42/latest_model.pt"),
    Path("/content/cafr_runs/imagenette/cafrnet_seed123/best_model.pt"),
    Path("/content/cafr_runs/imagenette/cafrnet_seed123/latest_model.pt")
]:
    print(p.name, "EXISTS" if p.exists() else "MISSING")


=== REVIEWER MATRIX LAUNCHER ===
Launcher not found at: /content/cartoonization-robustness-study/research/CAFRNet_v2/run_reviewer_matrix.sh

=== train_cafr_net.py ARGUMENTS ===
Not found: /content/cartoonization-robustness-study/research/CAFRNet_v2/train_cafr_net.py

=== train_architecture.py ARGUMENTS ===
Not found: /content/cartoonization-robustness-study/research/CAFRNet_v2/train_architecture.py

=== AVAILABLE CHECKPOINTS ===
best_model.pt EXISTS
latest_model.pt MISSING
best_model.pt EXISTS
latest_model.pt EXISTS


In [ ]:

from pathlib import Path

REPO = Path("/content/cartoonization-robustness-study")

print("=== TRAINING SCRIPTS ===")
for p in REPO.rglob("*.py"):
    if p.name in [
        "train_cafr_net.py",
        "train_architecture.py",
        "evaluate_cafr_net.py",
        "evaluate_architecture.py",
        "cafr_net.py"
    ]:
        print(p)

print("\n=== LAUNCHER FILES ===")
for p in REPO.rglob("*.sh"):
    if "reviewer" in p.name.lower() or "matrix" in p.name.lower():
        print(p)

print("\n=== SOURCE DIRECTORY ===")
for p in REPO.rglob("source"):
    if p.is_dir() and "CAFRNet_v2" in str(p):
        print(p)
        for f in sorted(p.iterdir()):
            print(" ", f.name)

print("\n=== CHECKPOINT STATUS ===")
for p in sorted(Path("/content/cafr_runs/imagenette").rglob("*.pt")):
    print(p, f"{p.stat().st_size/1024/1024:.1f} MB")


=== TRAINING SCRIPTS ===
/content/cartoonization-robustness-study/training/train_cafr_net.py
/content/cartoonization-robustness-study/evaluation/evaluate_cafr_net.py
/content/cartoonization-robustness-study/models/cafr_net.py
/content/cartoonization-robustness-study/research/CAFRNet_v2/source/train_cafr_net.py
/content/cartoonization-robustness-study/research/CAFRNet_v2/source/train_architecture.py
/content/cartoonization-robustness-study/research/CAFRNet_v2/source/evaluate_cafr_net.py
/content/cartoonization-robustness-study/research/CAFRNet_v2/source/cafr_net.py
/content/cartoonization-robustness-study/research/CAFRNet_v2/source/evaluate_architecture.py

=== LAUNCHER FILES ===
/content/cartoonization-robustness-study/research/CAFRNet_v2/source/run_reviewer_matrix.sh

=== SOURCE DIRECTORY ===
/content/cartoonization-robustness-study/research/CAFRNet_v2/source
  __pycache__
  analyze_architecture_comparison.py
  cafr_net.py
  evaluate_architecture.py
  evaluate_cafr_net.py
  legacy
  r

In [ ]:

import torch, json
from pathlib import Path

ROOT = Path("/content/cafr_runs/imagenette")

checkpoints = [
    ROOT / "cafrnet_seed42/best_model.pt",
    ROOT / "cafrnet_seed123/best_model.pt",
    ROOT / "cafrnet_seed123/latest_model.pt",
]

print("========== CAFR CHECKPOINT AUDIT ==========")
for path in checkpoints:
    print("\n", path)
    if not path.exists():
        print("MISSING")
        continue

    ck = torch.load(path, map_location="cpu", weights_only=False)
    print("Keys:", list(ck.keys()))
    print("Architecture:", ck.get("architecture"))
    print("Seed:", ck.get("seed"))
    print("Epoch:", ck.get("epoch"))

    meta = ck.get("metadata", {})
    protocol = meta.get("training_protocol", {})
    print("Batch size:", protocol.get("batch_size"))
    print("Train hash:", meta.get("train_manifest_sha256"))
    print("Validation hash:", meta.get("validation_manifest_sha256"))

print("\n========== EXISTING EXPERIMENT DIRECTORIES ==========")
for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print("\n", p.name)
        for f in ["best_model.pt", "latest_model.pt",
                  "run_metadata.json", "history.json"]:
            q = p / f
            if q.exists():
                print("  ", f, f"({q.stat().st_size/1024/1024:.1f} MB)")

print("\n========== TRAINING ENVIRONMENT ==========")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), "GB")


========== CAFR CHECKPOINT AUDIT ==========

 /content/cafr_runs/imagenette/cafrnet_seed42/best_model.pt
Keys: ['model', 'architecture', 'model_config', 'classes', 'seed', 'epoch', 'val_accuracy', 'metadata']
Architecture: cafrnet_v2
Seed: 42
Epoch: 3
Batch size: 16
Train hash: 39438146a850a0021c69e7d8fe215f20a6c883185f42e036387ff9627ca00d12
Validation hash: fd16a292e5810e9455d786f397ee69a2a8b514c7c29a2f51400673cbc3a36a6f

 /content/cafr_runs/imagenette/cafrnet_seed123/best_model.pt
Keys: ['model', 'architecture', 'model_config', 'classes', 'seed', 'epoch', 'val_accuracy', 'metadata']
Architecture: cafrnet_v2
Seed: 123
Epoch: 1
Batch size: 16
Train hash: 39438146a850a0021c69e7d8fe215f20a6c883185f42e036387ff9627ca00d12
Validation hash: fd16a292e5810e9455d786f397ee69a2a8b514c7c29a2f51400673cbc3a36a6f

 /content/cafr_runs/imagenette/cafrnet_seed123/latest_model.pt
Keys: ['model', 'architecture', 'model_config', 'classes', 'seed', 'epoch', 'best_accuracy', 'optimizer', 'scheduler', 'scaler

In [ ]:

import os
import sys
import torch
import urllib.request
import subprocess
from pathlib import Path

# ---------------- CONFIGURATION ----------------
REPO = Path("/content/cartoonization-robustness-study")
SRC = REPO / "research/CAFRNet_v2/source"
DATA = Path("/content/cafr_data_matched")
OUT = Path("/content/cafr_runs/imagenette")

TRAIN_HASH = "39438146a850a0021c69e7d8fe215f20a6c883185f42e036387ff9627ca00d12"
VAL_HASH = "fd16a292e5810e9455d786f397ee69a2a8b514c7c29a2f51400673cbc3a36a6f"

EPOCHS = 20
BATCH = 16
WORKERS = 2

assert (DATA / "train").is_dir()
assert (DATA / "val").is_dir()
assert (SRC / "train_architecture.py").exists()

os.chdir(SRC)

# ---------------- RESTORE SEED 42 FINAL CHECKPOINT ----------------
seed42_dir = OUT / "cafrnet_seed42"
seed42_dir.mkdir(parents=True, exist_ok=True)

seed42_latest = seed42_dir / "latest_model.pt"

if not seed42_latest.exists():
    url = (
        "https://github.com/Karthik6-cpu/"
        "cartoonization-robustness-study/releases/download/"
        "cafrnet-v2-checkpoints-2026-11-25/latest_model.pt"
    )
    tmp = seed42_dir / "latest_model.download"
    print("Downloading completed Seed 42 checkpoint...")
    urllib.request.urlretrieve(url, tmp)
    tmp.replace(seed42_latest)

ck = torch.load(seed42_latest, map_location="cpu", weights_only=False)

assert ck["architecture"] == "cafrnet_v2"
assert ck["seed"] == 42
assert ck["epoch"] == 20, f"Expected completed epoch 20, got {ck['epoch']}"
assert ck["metadata"]["train_manifest_sha256"] == TRAIN_HASH
assert ck["metadata"]["validation_manifest_sha256"] == VAL_HASH

print("Seed 42 verified: completed epoch", ck["epoch"])

# ---------------- DEFINE ONLY REQUIRED JOBS ----------------
jobs = []

# Resume existing CAFR-Net Seed 123
jobs.append(("cafrnet_v2", 123, "full", "cafrnet_seed123"))

# New CAFR-Net Seed 2026
jobs.append(("cafrnet_v2", 2026, "full", "cafrnet_seed2026"))

# ConvNeXt-Tiny: all three seeds
for seed in [42, 123, 2026]:
    jobs.append(("convnext_tiny", seed, "full",
                 f"convnext_tiny_seed{seed}"))

# ResNet-18: all three seeds
for seed in [42, 123, 2026]:
    jobs.append(("resnet18", seed, "full",
                 f"resnet18_seed{seed}"))

# Three ablations x three seeds
ablations = [
    ("no_edge", "cafrnet_no_edge"),
    ("no_semantic_fusion", "cafrnet_no_fusion"),
    ("no_attention", "cafrnet_no_attention"),
]

for ablation, prefix in ablations:
    for seed in [42, 123, 2026]:
        jobs.append((
            "cafrnet_v2", seed, ablation,
            f"{prefix}_seed{seed}"
        ))

print("\nJobs to execute:", len(jobs))
print("Completed Seed 42 CAFR-Net will NOT be retrained.")

# ---------------- SAFE SEQUENTIAL EXECUTION ----------------
for number, (arch, seed, ablation, name) in enumerate(jobs, 1):

    run_dir = OUT / name
    run_dir.mkdir(parents=True, exist_ok=True)

    latest = run_dir / "latest_model.pt"
    best = run_dir / "best_model.pt"

    print("\n" + "=" * 65)
    print(f"JOB {number}/{len(jobs)}: {name}")
    print("=" * 65)

    resume = False

    if latest.exists():
        old = torch.load(latest, map_location="cpu", weights_only=False)

        assert old.get("architecture") == arch, "Architecture mismatch"
        assert old.get("seed") == seed, "Seed mismatch"

        meta = old.get("metadata", {})
        assert meta.get("train_manifest_sha256") == TRAIN_HASH
        assert meta.get("validation_manifest_sha256") == VAL_HASH

        old_ablation = meta.get("arguments", {}).get("ablation", "full")
        assert old_ablation == ablation, "Ablation mismatch"

        if old["epoch"] >= EPOCHS:
            print("Already completed. Skipping.")
            continue

        resume = True
        print("Resuming from epoch", old["epoch"])

    elif best.exists():
        raise RuntimeError(
            f"{name}: best checkpoint exists without latest checkpoint. "
            "Stopping to protect existing results."
        )

    cmd = [
        sys.executable, "-u", "train_architecture.py",
        "--arch", arch,
        "--ablation", ablation,
        "--data", str(DATA),
        "--out", str(run_dir),
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH),
        "--workers", str(WORKERS),
    ]

    if resume:
        cmd.append("--resume")

    print("Command:", " ".join(cmd))

    subprocess.run(cmd, check=True)

    # Verify the job actually completed
    assert latest.exists(), f"Missing latest checkpoint for {name}"
    final_ck = torch.load(latest, map_location="cpu", weights_only=False)
    assert final_ck["epoch"] >= EPOCHS, (
        f"{name} stopped at epoch {final_ck['epoch']}"
    )
    assert best.exists(), f"Missing best checkpoint for {name}"

    print("COMPLETED:", name)

print("\nALL REQUESTED IMAGENETTE TRAINING JOBS FINISHED.")


Seed 42 verified: completed epoch 20

Jobs to execute: 17
Completed Seed 42 CAFR-Net will NOT be retrained.

JOB 1/17: cafrnet_seed123
Resuming from epoch 1
Command: /usr/bin/python3 -u train_architecture.py --arch cafrnet_v2 --ablation full --data /content/cafr_data_matched --out /content/cafr_runs/imagenette/cafrnet_seed123 --seed 123 --epochs 20 --batch-size 16 --workers 2 --resume
COMPLETED: cafrnet_seed123

JOB 2/17: cafrnet_seed2026
Command: /usr/bin/python3 -u train_architecture.py --arch cafrnet_v2 --ablation full --data /content/cafr_data_matched --out /content/cafr_runs/imagenette/cafrnet_seed2026 --seed 2026 --epochs 20 --batch-size 16 --workers 2
COMPLETED: cafrnet_seed2026

JOB 3/17: convnext_tiny_seed42
Command: /usr/bin/python3 -u train_architecture.py --arch convnext_tiny --ablation full --data /content/cafr_data_matched --out /content/cafr_runs/imagenette/convnext_tiny_seed42 --seed 42 --epochs 20 --batch-size 16 --workers 2


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/cartoonization-robustness-study")
RUNS = Path("/content/cafr_runs/imagenette")

print("=== REPOSITORY STATUS ===")
os.system(f"cd {ROOT} && git status --short")

print("\n=== EXPERIMENT FILES ===")
for base in [ROOT / "research/CAFRNet_v2", RUNS]:
    if base.exists():
        for f in sorted(base.rglob("*")):
            if f.is_file():
                print(f"{f} | {f.stat().st_size / (1024**2):.2f} MB")

=== REPOSITORY STATUS ===

=== EXPERIMENT FILES ===
/content/cartoonization-robustness-study/research/CAFRNet_v2/README.md | 0.01 MB
/content/cartoonization-robustness-study/research/CAFRNet_v2/RESULTS_SUMMARY.md | 0.01 MB
/content/cartoonization-robustness-study/research/CAFRNet_v2/REVIEWER_REVISION_PROTOCOL.md | 0.01 MB
/content/cartoonization-robustness-study/research/CAFRNet_v2/artifacts/CAFRNet_Final_Research_Package.zip | 1.16 MB
/content/cartoonization-robustness-study/research/CAFRNet_v2/colab_session_backup/20261124_161028/BACKUP_INVENTORY.json | 0.01 MB
/content/cartoonization-robustness-study/research/CAFRNet_v2/colab_session_backup/20261124_161028/CAFRNet_Final_Research_Package.zip | 1.16 MB
/content/cartoonization-robustness-study/research/CAFRNet_v2/colab_session_backup/20261124_161028/cafr_runs/CAFRNet_Final_Experimental_Results.zip | 0.27 MB
/content/cartoonization-robustness-study/research/CAFRNet_v2/colab_session_backup/20261124_161028/cafr_runs/CAFRNet_Final_Research

In [ ]:
import subprocess

result = subprocess.run(
    ["git", "ls-remote", "origin", "HEAD"],
    cwd="/content/cartoonization-robustness-study",
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("SUCCESS: GitHub authentication is working!")
    print(result.stdout)
else:
    print("GitHub authentication failed:")
    print(result.stderr)

SUCCESS: GitHub authentication is working!
0dbed5c589d2e239d9303d02e0d90e6d70cf3a8f	HEAD



In [ ]:
import subprocess

repo = "/content/cartoonization-robustness-study"
branch = "research/cafr-net-phase-iv"

result = subprocess.run(
    ["git", "push", "--dry-run", "origin", branch],
    cwd=repo,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode == 0:
    print("SUCCESS: GitHub push permission verified!")
else:
    print("Push permission needs attention.")


fatal: could not read Username for 'https://github.com': No such device or address

Push permission needs attention.


In [ ]:
import os
import subprocess
from getpass import getpass

REPO = "/content/cartoonization-robustness-study"

username = "Karthik6-cpu"
token = getpass("Paste your GitHub PAT (input remains hidden): ")

env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
env["GIT_CONFIG_VALUE_0"] = (
    "AUTHORIZATION: basic " +
    __import__("base64").b64encode(
        f"{username}:{token}".encode()
    ).decode()
)

result = subprocess.run(
    ["git", "push", "--dry-run", "origin", "research/cafr-net-phase-iv"],
    cwd=REPO,
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("PUSH TEST:", "SUCCESS" if result.returncode == 0 else "FAILED")

del token

Paste your GitHub PAT (input remains hidden): ··········

Everything up-to-date

PUSH TEST: SUCCESS


In [ ]:
from pathlib import Path
import shutil, zipfile, hashlib, json

RUNS = Path("/content/cafr_runs/imagenette")
REPO = Path("/content/cartoonization-robustness-study")
DEST = REPO / "research/CAFRNet_v2/controlled_runs/imagenette"
ASSETS = Path("/content/cafr_release_assets")

assert RUNS.exists(), "Experiment folder not found"

DEST.mkdir(parents=True, exist_ok=True)
ASSETS.mkdir(parents=True, exist_ok=True)

manifest = []

for run in sorted(RUNS.iterdir()):
    if not run.is_dir():
        continue

    print("Preserving:", run.name)

    # Copy logs, metadata, histories, manifests and other non-checkpoint files
    shutil.copytree(
        run,
        DEST / run.name,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns("*.pt", "*.pth", "*.ckpt")
    )

    # Preserve the complete run, including checkpoints and temporary files
    archive = ASSETS / f"{run.name}.zip"

    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:
        for f in run.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(RUNS))

    sha = hashlib.sha256()
    with open(archive, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            sha.update(chunk)

    manifest.append({
        "run": run.name,
        "archive": archive.name,
        "size_MB": round(archive.stat().st_size / 1024**2, 2),
        "sha256": sha.hexdigest()
    })

    print("Full archive:", archive.name)

manifest_path = (
    REPO / "research/CAFRNet_v2/release_asset_manifest.json"
)
manifest_path.write_text(json.dumps(manifest, indent=2))

print("\nPREPARATION COMPLETE")
print(json.dumps(manifest, indent=2))

Preserving: cafrnet_seed123
Full archive: cafrnet_seed123.zip
Preserving: cafrnet_seed2026
Full archive: cafrnet_seed2026.zip
Preserving: cafrnet_seed42
Full archive: cafrnet_seed42.zip
Preserving: convnext_tiny_seed42
Full archive: convnext_tiny_seed42.zip

PREPARATION COMPLETE
[
  {
    "run": "cafrnet_seed123",
    "archive": "cafrnet_seed123.zip",
    "size_MB": 462.84,
    "sha256": "626e2de928075bb207a6ed81976183072e28e5a31e95cc631beb269ae2ae8fc7"
  },
  {
    "run": "cafrnet_seed2026",
    "archive": "cafrnet_seed2026.zip",
    "size_MB": 462.84,
    "sha256": "9a2d09766b2ff659edf0b3fffb9789e5635242ee626641e8418ef5ed782b193b"
  },
  {
    "run": "cafrnet_seed42",
    "archive": "cafrnet_seed42.zip",
    "size_MB": 462.83,
    "sha256": "835464977f0d92f2f4487ab84d873fa9b617754edb3bd103f2ab28288dae3abf"
  },
  {
    "run": "convnext_tiny_seed42",
    "archive": "convnext_tiny_seed42.zip",
    "size_MB": 425.3,
    "sha256": "6ba016e1d35c3ff8a89f189e5d17cd834438bb1e691465cc226a402b

In [ ]:
import subprocess

REPO = "/content/cartoonization-robustness-study"

subprocess.run(
    ["git", "add", "research/CAFRNet_v2/controlled_runs/",
     "research/CAFRNet_v2/release_asset_manifest.json"],
    cwd=REPO, check=True
)

subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO, check=True
)

subprocess.run(
    ["git", "commit", "-m",
     "Preserve controlled Imagenette experiment files and logs"],
    cwd=REPO, check=True
)

p = subprocess.run(
    ["git", "push", "origin", "research/cafr-net-phase-iv"],
    cwd=REPO, env=env, capture_output=True, text=True
)

print(p.stdout)
print(p.stderr)
assert p.returncode == 0, "Git push failed"
print("SUCCESS: Experiment files pushed to GitHub")

CalledProcessError: Command '['git', 'commit', '-m', 'Preserve controlled Imagenette experiment files and logs']' returned non-zero exit status 128.

In [ ]:
import subprocess

REPO = "/content/cartoonization-robustness-study"

# Configure Git identity for this repository only
subprocess.run(
    ["git", "config", "user.name", "Karthik6-cpu"],
    cwd=REPO, check=True
)

subprocess.run(
    ["git", "config", "user.email",
     "157053962+Karthik6-cpu@users.noreply.github.com"],
    cwd=REPO, check=True
)

# Check staged files
subprocess.run(["git", "status", "--short"], cwd=REPO)

# Commit
commit = subprocess.run(
    ["git", "commit", "-m",
     "Preserve controlled Imagenette experiment files and logs"],
    cwd=REPO, capture_output=True, text=True
)

print(commit.stdout)
print(commit.stderr)

if commit.returncode == 0:
    print("COMMIT SUCCESSFUL")
else:
    print("COMMIT FAILED — inspect the error above")

[research/cafr-net-phase-iv 18e49e86] Preserve controlled Imagenette experiment files and logs
 18 files changed, 38477 insertions(+)
 create mode 100644 research/CAFRNet_v2/controlled_runs/imagenette/cafrnet_seed123/history.json
 create mode 100644 research/CAFRNet_v2/controlled_runs/imagenette/cafrnet_seed123/history.jsonl
 create mode 100644 research/CAFRNet_v2/controlled_runs/imagenette/cafrnet_seed123/run_metadata.json
 create mode 100644 research/CAFRNet_v2/controlled_runs/imagenette/cafrnet_seed123/train_split_manifest.csv
 create mode 100644 research/CAFRNet_v2/controlled_runs/imagenette/cafrnet_seed123/validation_split_manifest.csv
 create mode 100644 research/CAFRNet_v2/controlled_runs/imagenette/cafrnet_seed2026/history.json
 create mode 100644 research/CAFRNet_v2/controlled_runs/imagenette/cafrnet_seed2026/history.jsonl
 create mode 100644 research/CAFRNet_v2/controlled_runs/imagenette/cafrnet_seed2026/run_metadata.json
 create mode 100644 research/CAFRNet_v2/controlled_run

In [ ]:
import subprocess

p = subprocess.run(
    ["git", "push", "origin", "research/cafr-net-phase-iv"],
    cwd=REPO, env=env, capture_output=True, text=True
)

print(p.stdout)
print(p.stderr)
print("PUSH SUCCESSFUL" if p.returncode == 0 else "PUSH FAILED")


To https://github.com/Karthik6-cpu/cartoonization-robustness-study.git
   b39e73af..18e49e86  research/cafr-net-phase-iv -> research/cafr-net-phase-iv

PUSH SUCCESSFUL


In [ ]:
from pathlib import Path
import zipfile, hashlib, json

RUNS = Path("/content/cafr_runs/imagenette")
ASSETS = Path("/content/cafr_release_assets")

ASSETS.mkdir(parents=True, exist_ok=True)

manifest = []

for folder in sorted(RUNS.iterdir()):
    if not folder.is_dir():
        continue

    archive = ASSETS / f"{folder.name}_COMPLETE.zip"

    with zipfile.ZipFile(archive, "w", zipfile.ZIP_STORED) as z:
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(RUNS))

    digest = hashlib.sha256()
    with open(archive, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)

    manifest.append({
        "folder": folder.name,
        "archive": archive.name,
        "size_MB": round(archive.stat().st_size / 1024**2, 2),
        "sha256": digest.hexdigest()
    })

(ASSETS / "COMPLETE_BACKUP_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2)
)

print(json.dumps(manifest, indent=2))
print("Total archives:", len(manifest))

[
  {
    "folder": "cafrnet_seed123",
    "archive": "cafrnet_seed123_COMPLETE.zip",
    "size_MB": 462.84,
    "sha256": "626e2de928075bb207a6ed81976183072e28e5a31e95cc631beb269ae2ae8fc7"
  },
  {
    "folder": "cafrnet_seed2026",
    "archive": "cafrnet_seed2026_COMPLETE.zip",
    "size_MB": 462.84,
    "sha256": "9a2d09766b2ff659edf0b3fffb9789e5635242ee626641e8418ef5ed782b193b"
  },
  {
    "folder": "cafrnet_seed42",
    "archive": "cafrnet_seed42_COMPLETE.zip",
    "size_MB": 462.83,
    "sha256": "835464977f0d92f2f4487ab84d873fa9b617754edb3bd103f2ab28288dae3abf"
  },
  {
    "folder": "convnext_tiny_seed42",
    "archive": "convnext_tiny_seed42_COMPLETE.zip",
    "size_MB": 425.3,
    "sha256": "6ba016e1d35c3ff8a89f189e5d17cd834438bb1e691465cc226a402b52f3ecd2"
  }
]
Total archives: 4


In [ ]:
import os, base64, hashlib, requests
from pathlib import Path
from urllib.parse import quote
from tqdm.auto import tqdm

OWNER = "Karthik6-cpu"
REPO = "cartoonization-robustness-study"
TAG = "cafrnet-v2-checkpoints-2026-11-25"

ASSETS = Path("/content/cafr_release_assets")

# Recover token from the existing in-memory Git authentication header
encoded = env["GIT_CONFIG_VALUE_0"].split(" ", 1)[1]
PAT = base64.b64decode(encoded).decode().split(":", 1)[1]

headers = {
    "Authorization": f"token {PAT}",
    "Accept": "application/vnd.github+json"
}

# Find the existing release
url = f"https://api.github.com/repos/{OWNER}/{REPO}/releases/tags/{TAG}"
response = requests.get(url, headers=headers, timeout=60)
response.raise_for_status()
release = response.json()

upload_url = release["upload_url"].split("{", 1)[0]
existing = {a["name"]: a for a in release.get("assets", [])}

files = sorted(ASSETS.glob("*_COMPLETE.zip"))
files.append(ASSETS / "COMPLETE_BACKUP_MANIFEST.json")

assert len(files) == 5, f"Expected 4 ZIPs + manifest, found {len(files)}"

for path in files:
    assert path.exists(), f"Missing: {path}"

    name = path.name
    size = path.stat().st_size

    # Avoid duplicate uploads
    if name in existing:
        if existing[name]["size"] == size:
            print("Already uploaded and size verified:", name)
            continue
        raise RuntimeError(
            f"Asset name already exists with a different size: {name}"
        )

    print(f"\nUploading {name} ({size / 1024**2:.2f} MB)")

    progress = tqdm(total=size, unit="B", unit_scale=True)

    def file_chunks(file, chunk_size=8 * 1024 * 1024):
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            progress.update(len(chunk))
            yield chunk

    upload_headers = {
        **headers,
        "Content-Type": (
            "application/zip" if path.suffix == ".zip"
            else "application/json"
        ),
        "Content-Length": str(size)
    }

    with open(path, "rb") as f:
        result = requests.post(
            upload_url,
            params={"name": name},
            headers=upload_headers,
            data=file_chunks(f),
            timeout=(60, 3600)
        )

    progress.close()
    result.raise_for_status()

    asset = result.json()
    print("UPLOAD SUCCESS:", asset["name"])
    print("GitHub size:", asset["size"])

print("\nALL RELEASE ASSETS PROCESSED")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xab in position 1: invalid start byte

In [ ]:
import requests
from pathlib import Path
from getpass import getpass

OWNER = "Karthik6-cpu"
REPO = "cartoonization-robustness-study"
TAG = "cafrnet-v2-checkpoints-2026-11-25"

ASSETS = Path("/content/cafr_release_assets")

# Enter PAT securely
PAT = getpass("Enter GitHub PAT (hidden): ")

headers = {
    "Authorization": f"Bearer {PAT}",
    "Accept": "application/vnd.github+json"
}

# Locate existing release
api = f"https://api.github.com/repos/{OWNER}/{REPO}/releases/tags/{TAG}"
r = requests.get(api, headers=headers, timeout=60)
r.raise_for_status()

release = r.json()
upload_url = release["upload_url"].split("{")[0]

existing = {a["name"]: a for a in release.get("assets", [])}

files = sorted(ASSETS.glob("*_COMPLETE.zip"))
manifest = ASSETS / "COMPLETE_BACKUP_MANIFEST.json"

if manifest.exists():
    files.append(manifest)

assert len(files) == 5, f"Expected 5 files, found {len(files)}"

for path in files:
    name = path.name
    size = path.stat().st_size

    if name in existing:
        if existing[name]["size"] == size:
            print("Already uploaded:", name)
            continue
        raise RuntimeError(f"Existing asset has different size: {name}")

    print(f"\nUploading {name} ({size / 1024**2:.2f} MB)")

    upload_headers = {
        **headers,
        "Content-Type": (
            "application/zip" if path.suffix == ".zip"
            else "application/json"
        ),
        "Content-Length": str(size)
    }

    with open(path, "rb") as f:
        result = requests.post(
            upload_url,
            params={"name": name},
            headers=upload_headers,
            data=f,
            timeout=(60, 3600)
        )

    if result.status_code not in (200, 201):
        print(result.text[:2000])
        result.raise_for_status()

    print("SUCCESS:", result.json()["name"])

print("\nRELEASE UPLOAD PROCESS FINISHED")
del PAT

Enter GitHub PAT (hidden): ··········

Uploading cafrnet_seed123_COMPLETE.zip (462.84 MB)
SUCCESS: cafrnet_seed123_COMPLETE.zip

Uploading cafrnet_seed2026_COMPLETE.zip (462.84 MB)
SUCCESS: cafrnet_seed2026_COMPLETE.zip

Uploading cafrnet_seed42_COMPLETE.zip (462.83 MB)
SUCCESS: cafrnet_seed42_COMPLETE.zip

Uploading convnext_tiny_seed42_COMPLETE.zip (425.30 MB)
SUCCESS: convnext_tiny_seed42_COMPLETE.zip

Uploading COMPLETE_BACKUP_MANIFEST.json (0.00 MB)
SUCCESS: COMPLETE_BACKUP_MANIFEST.json

RELEASE UPLOAD PROCESS FINISHED


In [ ]:
from pathlib import Path

print("=== NOTEBOOK FILES ===")
for p in Path("/content").rglob("*.ipynb"):
    if ".ipynb_checkpoints" not in str(p):
        print(p, round(p.stat().st_size / 1024**2, 2), "MB")

print("\n=== CAFR RESEARCH ZIP FILES ===")
root = Path("/content/cartoonization-robustness-study/research/CAFRNet_v2")

for p in root.rglob("*.zip"):
    print(p, round(p.stat().st_size / 1024**2, 2), "MB")

print("\n=== TOP-LEVEL CONTENTS ===")
for p in Path("/content").iterdir():
    print(p.name, "DIR" if p.is_dir() else "FILE")

=== NOTEBOOK FILES ===
/content/cartoonization-robustness-study/notebooka218aec394.ipynb 2.13 MB
/content/cartoonization-robustness-study/01_opencv_cartoonization_pilot (1).ipynb 45.62 MB
/content/cartoonization-robustness-study/notebooka218aec394 (2).ipynb 2.55 MB
/content/cartoonization-robustness-study/01_opencv_cartoonization_pilot.ipynb 45.62 MB
/content/cartoonization-robustness-study/convnext.ipynb 0.2 MB
/content/cartoonization-robustness-study/Cartoonization_JPEG_Robustness_Research.ipynb 0.69 MB
/content/cartoonization-robustness-study/convnext_next_steps.ipynb 0.17 MB
/content/cartoonization-robustness-study/notebooks/CAFRNet_Phase_IV_Research.ipynb 1.59 MB
/content/cartoonization-robustness-study/notebooks/CAFRNet_v2_Final_Research_Experiments.ipynb 0.14 MB
/content/cartoonization-robustness-study/notebooks/CAFRNet_v2_Final_Research_Experiments_2.ipynb 0.16 MB
/content/cartoonization-robustness-study/BICR_Net_Research/bicr-net-experiments.ipynb 6.18 MB
/content/cartoonizati

In [ ]:
from pathlib import Path
import subprocess
import os

REPO = Path("/content/cartoonization-robustness-study")

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO,
        capture_output=True,
        text=True
    )

print("========== NOTEBOOK STATUS ==========")

for f in sorted(REPO.rglob("*.ipynb")):
    if ".ipynb_checkpoints" in str(f):
        continue

    rel = str(f.relative_to(REPO))
    tracked = git("ls-files", "--error-unmatch", rel)

    status = "ALREADY TRACKED" if tracked.returncode == 0 else "NOT TRACKED"

    print(
        status,
        "|",
        rel,
        "|",
        round(f.stat().st_size / 1024**2, 2),
        "MB"
    )

print("\n========== GIT STATUS ==========")
print(git("status", "--short", "--untracked-files=all").stdout)

print("\n========== WORKSPACE SIZES ==========")

for name in [
    "cafr_data_matched",
    "cafr_data",
    "imagenette2",
    "cafr_runs",
    "cafr_release_assets",
    "sample_data"
]:
    p = Path("/content") / name

    if p.exists():
        total = sum(
            f.stat().st_size
            for f in p.rglob("*")
            if f.is_file()
        )
        print(name, ":", round(total / 1024**3, 3), "GB")

print("\n========== OTHER TEMPORARY FILES ==========")

for p in Path("/content").iterdir():
    if p.is_file():
        print(p.name, round(p.stat().st_size / 1024**2, 2), "MB")

========== NOTEBOOK STATUS ==========
ALREADY TRACKED | 01_opencv_cartoonization_pilot (1).ipynb | 45.62 MB
ALREADY TRACKED | 01_opencv_cartoonization_pilot.ipynb | 45.62 MB
ALREADY TRACKED | BICR_Net_Research/bicr-net-experiments.ipynb | 6.18 MB
ALREADY TRACKED | Cartoonization_JPEG_Robustness_Research.ipynb | 0.69 MB
ALREADY TRACKED | convnext.ipynb | 0.2 MB
ALREADY TRACKED | convnext_next_steps.ipynb | 0.17 MB
ALREADY TRACKED | notebooka218aec394 (2).ipynb | 2.55 MB
ALREADY TRACKED | notebooka218aec394.ipynb | 2.13 MB
ALREADY TRACKED | notebooks/CAFRNet_Phase_IV_Research.ipynb | 1.59 MB
ALREADY TRACKED | notebooks/CAFRNet_v2_Final_Research_Experiments.ipynb | 0.14 MB
ALREADY TRACKED | notebooks/CAFRNet_v2_Final_Research_Experiments_2.ipynb | 0.16 MB
ALREADY TRACKED | research/CAFRNet_v2/notebooks/CAFRNet_v2_Reviewer_Revision_Experiments_Backup_2026-11-25.ipynb | 0.64 MB
ALREADY TRACKED | research_extension/compound_corruption_gpu_one_cell.ipynb | 0.01 MB

========== GIT STATUS =====

In [ ]:
from pathlib import Path
import hashlib

folders = [
    Path("/content/cafr_runs"),
    Path("/content/cafr_release_assets"),
    Path("/content/cartoonization-robustness-study/research/CAFRNet_v2"),
]

for folder in folders:
    print(f"\n{'='*70}\n{folder}\n{'='*70}")

    if not folder.exists():
        continue

    for f in sorted(folder.rglob("*")):
        if f.is_file():
            print(
                f"{f.relative_to(folder)}"
                f" | {f.stat().st_size / 1024**2:.2f} MB"
            )

print("\n=== MATRIX LOG ===")
p = Path("/content/imagenette_matrix.log")
if p.exists():
    print(p.read_text(errors="replace")[-10000:])


/content/cafr_runs
imagenette/cafrnet_seed123/best_model.pt | 115.61 MB
imagenette/cafrnet_seed123/history.json | 0.00 MB
imagenette/cafrnet_seed123/history.jsonl | 0.00 MB
imagenette/cafrnet_seed123/latest_model.pt | 346.84 MB
imagenette/cafrnet_seed123/run_metadata.json | 0.00 MB
imagenette/cafrnet_seed123/train_split_manifest.csv | 0.32 MB
imagenette/cafrnet_seed123/validation_split_manifest.csv | 0.06 MB
imagenette/cafrnet_seed2026/best_model.pt | 115.61 MB
imagenette/cafrnet_seed2026/history.json | 0.00 MB
imagenette/cafrnet_seed2026/history.jsonl | 0.00 MB
imagenette/cafrnet_seed2026/latest_model.pt | 346.84 MB
imagenette/cafrnet_seed2026/run_metadata.json | 0.00 MB
imagenette/cafrnet_seed2026/train_split_manifest.csv | 0.32 MB
imagenette/cafrnet_seed2026/validation_split_manifest.csv | 0.06 MB
imagenette/cafrnet_seed42/best_model.pt | 115.61 MB
imagenette/cafrnet_seed42/latest_model.pt | 346.84 MB
imagenette/cafrnet_seed42/run_metadata.json | 0.00 MB
imagenette/cafrnet_seed42/t

In [ ]:

from pathlib import Path
from datetime import datetime
import os

ROOT = Path("/content")

print("========== DATASET INVENTORY ==========")

for name in ["cafr_data_matched", "cafr_data", "imagenette2"]:
    folder = ROOT / name
    if folder.exists():
        files = [p for p in folder.rglob("*") if p.is_file()]
        total = sum(p.stat().st_size for p in files)
        print(f"\n{name}")
        print("Files:", len(files))
        print("Size: %.2f GB" % (total / 1024**3))
        print("Examples:", [str(p.relative_to(folder)) for p in files[:5]])

print("\n========== ORIGINAL DATASET ARCHIVE ==========")
p = ROOT / "imagenette2.tgz"
if p.exists():
    print("Size: %.2f GB" % (p.stat().st_size / 1024**3))

print("\n========== LATEST NOTEBOOKS ==========")
repo = ROOT / "cartoonization-robustness-study"
notebooks = sorted(
    repo.rglob("*.ipynb"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

for p in notebooks[:8]:
    modified = datetime.fromtimestamp(p.stat().st_mtime)
    print(modified.strftime("%Y-%m-%d %H:%M:%S"),
          p.relative_to(repo))

print("\n========== TEMPORARY LOG ==========")
log = ROOT / "imagenette_matrix.log"
if log.exists():
    print(log.read_text(errors="replace")[-3000:])


========== DATASET INVENTORY ==========

cafr_data_matched
Files: 14815
Size: 1.62 GB
Examples: ['val/n02102040/n02102040_7626.JPEG', 'val/n02102040/n02102040_1249.JPEG', 'val/n02102040/n02102040_584.JPEG', 'val/n02102040/n02102040_1675.JPEG', 'val/n02102040/n02102040_2149.JPEG']

cafr_data
Files: 13394
Size: 1.46 GB
Examples: ['test_original/n02102040/n02102040_251.JPEG', 'test_original/n02102040/n02102040_362.JPEG', 'test_original/n02102040/n02102040_5850.JPEG', 'test_original/n02102040/n02102040_5881.JPEG', 'test_original/n02102040/n02102040_1241.JPEG']

imagenette2
Files: 13395
Size: 1.46 GB
Examples: ['noisy_imagenette.csv', 'val/n02102040/n02102040_251.JPEG', 'val/n02102040/n02102040_362.JPEG', 'val/n02102040/n02102040_5850.JPEG', 'val/n02102040/n02102040_5881.JPEG']

========== ORIGINAL DATASET ARCHIVE ==========
Size: 1.45 GB

========== LATEST NOTEBOOKS ==========
2026-11-25 13:11:38 research_extension/compound_corruption_gpu_one_cell.ipynb
2026-11-25 13:11:37 research/CAFRNet

In [ ]:

from pathlib import Path
import filecmp

base = Path("/content")
folders = ["cafr_data", "imagenette2", "cafr_data_matched"]

def compare_folders(a, b):
    result = filecmp.dircmp(base/a, base/b)
    same = []
    different = []
    only_a = []
    only_b = []

    def walk(d, prefix=""):
        same.extend(prefix+x for x in d.same_files)
        different.extend(prefix+x for x in d.diff_files)
        only_a.extend(prefix+x for x in d.left_only)
        only_b.extend(prefix+x for x in d.right_only)
        for name, sub in d.subdirs.items():
            walk(sub, prefix+name+"/")

    walk(result)
    return same, different, only_a, only_b

for a, b in [("cafr_data", "imagenette2")]:
    same, different, only_a, only_b = compare_folders(a, b)
    print(f"\n{a} vs {b}")
    print("Identical files:", len(same))
    print("Different files:", len(different))
    print(f"Only in {a}:", len(only_a))
    print(f"Only in {b}:", len(only_b))
    print("Examples only in imagenette2:", only_b[:10])



cafr_data vs imagenette2
Identical files: 8516
Different files: 0
Only in cafr_data: 954
Only in imagenette2: 4879
Examples only in imagenette2: ['noisy_imagenette.csv', 'train/n01440764/ILSVRC2012_val_00026397.JPEG', 'train/n01440764/ILSVRC2012_val_00037596.JPEG', 'train/n01440764/ILSVRC2012_val_00048969.JPEG', 'train/n01440764/n01440764_10307.JPEG', 'train/n01440764/n01440764_10365.JPEG', 'train/n01440764/n01440764_10444.JPEG', 'train/n01440764/n01440764_105.JPEG', 'train/n01440764/n01440764_10555.JPEG', 'train/n01440764/n01440764_10716.JPEG']


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/imagenette_matrix.log")
dst = Path(
    "/content/cartoonization-robustness-study/"
    "research/CAFRNet_v2/imagenette_matrix.log"
)

if src.exists():
    shutil.copy2(src, dst)
    print("Log copied:", dst)

Log copied: /content/cartoonization-robustness-study/research/CAFRNet_v2/imagenette_matrix.log


In [ ]:

from pathlib import Path
from getpass import getpass
import zipfile
import hashlib
import requests
import json
import time

ROOT = Path("/content")
ASSETS = ROOT / "cafr_release_assets"
ASSETS.mkdir(exist_ok=True)

OWNER = "Karthik6-cpu"
REPO = "cartoonization-robustness-study"
TAG = "cafrnet-v2-checkpoints-2026-11-25"

# Full dataset snapshots plus original source archive
jobs = [
    ("cafr_data_matched", ROOT / "cafr_data_matched",
     ASSETS / "cafr_data_matched_2026-11-25.zip"),
    ("cafr_data", ROOT / "cafr_data",
     ASSETS / "cafr_data_2026-11-25.zip"),
    ("imagenette2", ROOT / "imagenette2",
     ASSETS / "imagenette2_2026-11-25.zip"),
]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "backup_date": "2026-11-25",
    "repository": f"{OWNER}/{REPO}",
    "release_tag": TAG,
    "assets": []
}

# Create full, lossless ZIP snapshots
for name, folder, archive in jobs:
    if not folder.exists():
        raise FileNotFoundError(folder)

    print(f"\nCreating {archive.name} ...")
    if archive.exists():
        archive.unlink()

    with zipfile.ZipFile(
        archive, "w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True
    ) as z:
        files = [p for p in folder.rglob("*") if p.is_file()]
        for i, p in enumerate(files, 1):
            z.write(p, Path(name) / p.relative_to(folder))
            if i % 2000 == 0:
                print(f"  Archived {i}/{len(files)} files")

    size = archive.stat().st_size
    if size >= 2 * 1024**3:
        raise RuntimeError(
            f"{archive.name} exceeds GitHub's 2 GB per-asset limit"
        )

    print("Files:", len(files))
    print("Size: %.2f GB" % (size / 1024**3))

    manifest["assets"].append({
        "name": archive.name,
        "size_bytes": size,
        "sha256": sha256(archive),
        "source_folder": str(folder)
    })

# Include original downloaded dataset archive
original = ROOT / "imagenette2.tgz"
if not original.exists():
    raise FileNotFoundError(original)

manifest["assets"].append({
    "name": "imagenette2_original_2026-11-25.tgz",
    "size_bytes": original.stat().st_size,
    "sha256": sha256(original),
    "source_file": str(original)
})

manifest_path = ASSETS / "DATASET_BACKUP_MANIFEST_2026-11-25.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

# GitHub Release authentication
PAT = getpass("Enter GitHub PAT (input hidden): ")
headers = {
    "Authorization": f"Bearer {PAT}",
    "Accept": "application/vnd.github+json"
}

api = f"https://api.github.com/repos/{OWNER}/{REPO}/releases/tags/{TAG}"
response = requests.get(api, headers=headers, timeout=60)
response.raise_for_status()

release = response.json()
upload_url = release["upload_url"].split("{")[0]
existing = {a["name"] for a in release.get("assets", [])}

upload_jobs = [
    (archive, archive.name)
    for _, _, archive in jobs
]
upload_jobs.append((original, "imagenette2_original_2026-11-25.tgz"))
upload_jobs.append((manifest_path, manifest_path.name))

for path, remote_name in upload_jobs:
    if remote_name in existing:
        print("ALREADY EXISTS — skipping:", remote_name)
        continue

    print(f"\nUploading {remote_name} ...")
    with open(path, "rb") as f:
        r = requests.post(
            upload_url,
            params={"name": remote_name},
            headers={
                **headers,
                "Content-Type": "application/octet-stream"
            },
            data=f,
            timeout=(60, 3600)
        )

    if r.status_code not in (200, 201):
        print("UPLOAD FAILED:", r.status_code, r.text[:1000])
        r.raise_for_status()

    print("UPLOAD SUCCESS:", remote_name)

print("\nDATASET UPLOAD PROCESS FINISHED.")
print("Release:", f"https://github.com/{OWNER}/{REPO}/releases/tag/{TAG}")



Creating cafr_data_matched_2026-11-25.zip ...
  Archived 2000/14815 files
  Archived 4000/14815 files
  Archived 6000/14815 files
  Archived 8000/14815 files
  Archived 10000/14815 files
  Archived 12000/14815 files
  Archived 14000/14815 files
Files: 14815
Size: 1.62 GB

Creating cafr_data_2026-11-25.zip ...
  Archived 2000/13394 files
  Archived 4000/13394 files
  Archived 6000/13394 files
  Archived 8000/13394 files
  Archived 10000/13394 files
  Archived 12000/13394 files
Files: 13394
Size: 1.46 GB

Creating imagenette2_2026-11-25.zip ...
  Archived 2000/13395 files
  Archived 4000/13395 files
  Archived 6000/13395 files
  Archived 8000/13395 files
  Archived 10000/13395 files
  Archived 12000/13395 files
Files: 13395
Size: 1.46 GB
Enter GitHub PAT (input hidden): ··········

Uploading cafr_data_matched_2026-11-25.zip ...
UPLOAD SUCCESS: cafr_data_matched_2026-11-25.zip

Uploading cafr_data_2026-11-25.zip ...
UPLOAD SUCCESS: cafr_data_2026-11-25.zip

Uploading imagenette2_2026-11-

In [ ]:

import subprocess
from pathlib import Path

REPO_DIR = Path("/content/cartoonization-robustness-study")

subprocess.run(
    ["git", "add", "research/CAFRNet_v2/imagenette_matrix.log"],
    cwd=REPO_DIR, env=env, check=True
)

status = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_DIR, env=env,
    capture_output=True, text=True, check=True
)
print(status.stdout)

if status.stdout.strip():
    subprocess.run(
        ["git", "commit", "-m", "Preserve Imagenette experiment matrix log"],
        cwd=REPO_DIR, env=env, check=True
    )
    subprocess.run(
        ["git", "push", "origin", "research/cafr-net-phase-iv"],
        cwd=REPO_DIR, env=env, check=True
    )
else:
    print("No new Git changes to push.")

print("Log push check finished.")


A  research/CAFRNet_v2/imagenette_matrix.log

Log push check finished.


In [ ]:
import subprocess
from pathlib import Path

repo = Path("/content/cartoonization-robustness-study")

print("GIT STATUS:")
subprocess.run(["git", "status", "--short"], cwd=repo)

print("\nLATEST COMMIT:")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=repo)

print("\nLOG EXISTS:")
print((repo / "research/CAFRNet_v2/imagenette_matrix.log").exists())

GIT STATUS:

LATEST COMMIT:

LOG EXISTS:
True
